<a href="https://colab.research.google.com/github/vinaykz/devopsdemo/blob/main/ML104/SigmoidDeadAhead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Initializing

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
file_string = "/content/drive/My Drive/goodtitanic.csv"
data = pd.read_csv(file_string)
data

,pclass,survived,sex,age,sibsp,parch,embarked
0,1.0,1.0,female,29.0000,0.0,0.0,S
1,1.0,1.0,male,0.9167,1.0,2.0,S
2,1.0,0.0,female,2.0000,1.0,2.0,S
3,1.0,0.0,male,30.0000,1.0,2.0,S
4,1.0,0.0,female,25.0000,1.0,2.0,S
...,...,...,...,...,...,...,...
1304,3.0,0.0,female,14.5000,1.0,0.0,C
1305,3.0,0.0,female,NaN,1.0,0.0,C
1306,3.0,0.0,male,26.5000,0.0,0.0,C
1307,3.0,0.0,male,27.0000,0.0,0.0,C


## Exploration

###Key to columns:

`pclass`: passenger class (1-3)

`sibsp`: number of siblings or spouses aboard

`parch`: number of parents or children aboard

`fare`: passenger fare

`embarked`:  port of embarkation (C = Cherbourg; Q = Queenstown; S = Southampton)




In [6]:
data.describe(include='all')

,pclass,survived,sex,age,sibsp,parch,embarked
count,1309.000000,1309.000000,1309,1046.000000,1309.000000,1309.000000,1307
unique,NaN,NaN,2,NaN,NaN,NaN,3
top,NaN,NaN,male,NaN,NaN,NaN,S
freq,NaN,NaN,843,NaN,NaN,NaN,914
mean,2.294882,0.381971,NaN,29.881135,0.498854,0.385027,NaN
std,0.837836,0.486055,NaN,14.413500,1.041658,0.865560,NaN
min,1.000000,0.000000,NaN,0.166700,0.000000,0.000000,NaN
25%,2.000000,0.000000,NaN,21.000000,0.000000,0.000000,NaN
50%,3.000000,0.000000,NaN,28.000000,0.000000,0.000000,NaN
75%,3.000000,1.000000,NaN,39.000000,1.000000,0.000000,NaN


In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   pclass    1309 non-null   float64
 1   survived  1309 non-null   float64
 2   sex       1309 non-null   object 
 3   age       1046 non-null   float64
 4   sibsp     1309 non-null   float64
 5   parch     1309 non-null   float64
 6   embarked  1307 non-null   object 
dtypes: float64(5), object(2)
memory usage: 71.7+ KB


## Preprocessing

In [8]:
data.isnull().sum()

,0
pclass,0
survived,0
sex,0
age,263
sibsp,0
parch,0
embarked,2


In [21]:
# Remove samples (rows) if NaN values are present.
print(f"Original dataset shape: {data.shape}")
print(f"NaN values before dropna:\n{data.isnull().sum()[data.isnull().sum() > 0]}")
data = data.dropna(axis=0) # axis=0 specifies we're operating on rows
print(f"The dataset's shape after dropping NaN values: {data.shape}")
print(f"NaN values after dropna:\n{data.isnull().sum()[data.isnull().sum() > 0]}")

Original dataset shape: (1309, 8)
NaN values before dropna:
age    263
dtype: int64
The dataset's shape after dropping NaN values: (1046, 8)
NaN values after dropna:
Series([], dtype: int64)


In [23]:
data = pd.get_dummies(data, columns=['sex', 'embarked'], drop_first=True, dtype=int)
# The previous line with drop_first=True already handles dropping one of the dummy variables (e.g., 'sex_male')
# to avoid multicollinearity. Therefore, the explicit drop for 'sex_male' is no longer needed.
# data = data.drop(columns=['sex_male'])
data.head()

KeyError: "None of [Index(['sex', 'embarked'], dtype='object')] are in the [columns]"

In [24]:
# first split x and y
X = data.drop("survived", axis=1)
y = data.loc[:, "survived"]

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)
print(f"X train length: {len(X_train)}, Y train length: {len(y_train)}")
print(f"X test length: {len(X_test)}, Y test length {len(y_test)}")

X train length: 732, Y train length: 732
X test length: 314, Y test length 314


## Training

In [26]:
from sklearn.linear_model import LogisticRegression
log_reg_model = LogisticRegression(max_iter=500)
log_reg_model.fit(X_train, y_train)

LogisticRegression(max_iter=500)

## Testing

In [27]:
print(log_reg_model.score(X_test, y_test))

0.7452229299363057


## Deployment

In [29]:
leo = pd.DataFrame({"pclass": [3.0], "age": [23.0000], "sibsp": [0.0],
                    "parch": [0.0], "sex_male": [1], # Leo is male
                    "embarked_Q": [0], "embarked_S": [0]}) # Leo embarked at C (implied)
kate = pd.DataFrame({"pclass": [1.0], "age": [22.0000], "sibsp": [1.0],
                     "parch": [1.0], "sex_male": [0], # Kate is female
                     "embarked_Q": [1], "embarked_S": [0]}) # Kate embarked at Q

leos_fate = log_reg_model.predict(leo)[0]
kates_fate = log_reg_model.predict(kate)[0]

In [30]:
print(leos_fate)

0.0


In [31]:
print(kates_fate)

1.0
